# Future Climate — Annual Projection Plots

Broken-axis annual projection plots (observed 1985–2015 next to projected
2026–2098, SSP2-4.5 vs SSP5-8.5 ensemble median + 10–90% spread) for
precipitation, Tmax, and Tmin, per station.

**Reads:** historical annual files from `1_historical_climate_updated.ipynb`
(`Historical/Annual/`) and future annual files from `2_future_climate_data.ipynb`
(`New_Future_Climate/{variable}/station_data_annual/`).

**Writes:** one PNG per station per variable, under
`New_Future_Climate/{ppt: "Annual Rainfall Plots", tmax/tmin: "Annual {variable} Plots"}/`.

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_style("whitegrid")
sns.set_context("talk")


## Shared plotting function

The precipitation and temperature sections used to each have their own
~150-line copy of this exact plot (only the labels, historical column, and
output path differed). Both now call this one function instead.

In [ ]:
def plot_broken_axis_projection(station, future_path, historical_path, ylabel, title, out_path,
                                 hist_range=(1985, 2015)):
    """Two-panel broken-axis plot: observed (left) next to SSP2-4.5 / SSP5-8.5
    ensemble median + 10-90% spread (right), for one station."""
    future = pd.read_csv(future_path)
    historical = pd.read_csv(historical_path)

    hist = historical[["Year", station]].copy()
    hist = hist[(hist.Year >= hist_range[0]) & (hist.Year <= hist_range[1])]

    models = future.columns.drop("Year")
    models_245 = [m for m in models if m.startswith("245")]
    models_585 = [m for m in models if m.startswith("585")]

    future["Median_245"] = future[models_245].median(axis=1)
    future["Median_585"] = future[models_585].median(axis=1)
    future["P10_245"] = future[models_245].quantile(0.10, axis=1)
    future["P90_245"] = future[models_245].quantile(0.90, axis=1)
    future["P10_585"] = future[models_585].quantile(0.10, axis=1)
    future["P90_585"] = future[models_585].quantile(0.90, axis=1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8),
                                   gridspec_kw={"width_ratios": [1.3, 3]}, sharey=True)

    # LEFT: Observed
    ax1.plot(hist.Year, hist[station], color="black", lw=2.5, marker="o", ms=4, label="Observed")
    ax1.set_xlim(*hist_range)
    ax1.set_title(f"Observed ({hist_range[0]}\u2013{hist_range[1]})", fontsize=15, fontweight="bold")
    ax1.set_ylabel(ylabel, fontsize=18)
    ax1.grid(True, alpha=0.3)

    # RIGHT: Projected
    ax2.axvspan(2026, 2051, color="green", alpha=.03)
    ax2.axvspan(2051, 2076, color="orange", alpha=.03)
    ax2.axvspan(2076, 2099, color="red", alpha=.03)

    ax2.plot(future.Year, future.Median_245, color="steelblue", lw=2.5, linestyle="--", label="SSP2-4.5 median")
    ax2.plot(future.Year, future.Median_585, color="darkred", lw=2.5, linestyle="-", label="SSP5-8.5 median")
    ax2.fill_between(future.Year, future.P10_245, future.P90_245, color="lightblue", alpha=0.4, label="SSP2-4.5 spread")
    ax2.fill_between(future.Year, future.P10_585, future.P90_585, color="salmon", alpha=0.25, label="SSP5-8.5 spread")

    ax2.legend(loc="upper left", fontsize=12, frameon=True, facecolor="white", edgecolor="gray",
               bbox_to_anchor=(-0.25, 0.95), bbox_transform=ax2.transAxes, borderaxespad=0.0)

    ax2.set_xlim(2026, 2098)
    ax2.set_title("Projected (2026\u20132098)", fontsize=15, fontweight="bold")
    ax2.grid(True, alpha=.3)

    ylim = ax2.get_ylim()
    for x, label, color in [
        (2038, "Near Future\n(2026\u20132050)", "green"),
        (2063, "Mid Future\n(2051\u20132075)", "darkorange"),
        (2088, "Far Future\n(2076\u20132098)", "darkred"),
    ]:
        ax2.text(x, ylim[1] * 0.98, label, color=color, ha="center", va="top", fontsize=12, fontweight="bold")

    # Broken-axis marks
    ax1.spines["right"].set_visible(False)
    ax2.spines["left"].set_visible(False)
    ax2.tick_params(left=False)
    ax2.yaxis.tick_right()
    ax2.tick_params(labelright=False)

    d = .015
    kwargs = dict(transform=ax1.transAxes, color="k", clip_on=False)
    ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)
    ax1.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
    kwargs.update(transform=ax2.transAxes)
    ax2.plot((-d, +d), (-d, +d), **kwargs)
    ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)

    fig.suptitle(f"{title} - Station {station}", fontsize=20, fontweight="bold")
    fig.supxlabel("Year", fontsize=18)

    plt.subplots_adjust(wspace=0.05, right=0.88, top=0.90)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.show()


## Precipitation

In [ ]:
annual_historical_pcp = pd.read_csv('../All_DATA/Climate/Historical/Annual/historical_yearly_pcp.csv')
pcp_stations = annual_historical_pcp.columns[1:]
pcp_stations


In [ ]:
PCP_PLOT_DIR = '../All_DATA/Climate/New_Future_Climate/ppt/Annual Rainfall Plots'

for station in pcp_stations:
    plot_broken_axis_projection(
        station,
        future_path=f'../All_DATA/Climate/New_Future_Climate/ppt/station_data_annual/{station}_annual_future.csv',
        historical_path='../All_DATA/Climate/Historical/Annual/historical_yearly_pcp.csv',
        ylabel='Annual Rainfall (mm)',
        title='Annual Rainfall Projection',
        out_path=f'{PCP_PLOT_DIR}/Annual_Rainfall_Projection_{station}.png',
    )


## Temperature (Tmax / Tmin)

In [ ]:
historical_tmax = pd.read_csv('../All_DATA/Climate/Historical/Annual/tmax_historical_yearly_aggregation.csv')
historical_tmin = pd.read_csv('../All_DATA/Climate/Historical/Annual/tmin_historical_yearly_aggregation.csv')
temp_stations = historical_tmax.columns[1:].to_list()

TEMP_VARIABLES = ['tmax', 'tmin']
TEMP_TITLES = {'tmax': 'Annual Maximum Temperature', 'tmin': 'Annual Minimum Temperature'}
TEMP_YLABEL = 'Temperature (°C)'

temp_stations


In [ ]:
for variable in TEMP_VARIABLES:
    plot_dir = f'../All_DATA/Climate/New_Future_Climate/{variable}/Annual {variable} Plots'
    for station in temp_stations:
        plot_broken_axis_projection(
            station,
            future_path=f'../All_DATA/Climate/New_Future_Climate/{variable}/station_data_annual/{station}_annual_future.csv',
            historical_path=f'../All_DATA/Climate/Historical/Annual/{variable}_historical_yearly_aggregation.csv',
            ylabel=TEMP_YLABEL,
            title=TEMP_TITLES[variable],
            out_path=f'{plot_dir}/Annual_{variable}_Projection_{station}.png',
        )
